In [0]:
%run "../../commons/commons_imports"

In [0]:
df_aluno_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_ALUNO,
    recursive_by_year=False,
    format = "delta",
)

In [0]:
df_aluno_gold = (

    df_aluno_silver
    # =====================================================
    # Faixa Proficiencia
    # =====================================================
    .withColumn(
    "FAIXA_PROFICIENCIA",
    when(col("VL_PROFICIENCIA_LP").isNull(), "Não avaliado")
    .when(col("VL_PROFICIENCIA_LP") < 650, "<650")
    .when(col("VL_PROFICIENCIA_LP") < 700, "650-699")
    .when(col("VL_PROFICIENCIA_LP") < 743, "700-742")
    .when(col("VL_PROFICIENCIA_LP") < 800, "743-799")
    .otherwise("800+")
    )
    # =====================================================
    # Participação na avaliação
    # =====================================================

    .withColumn(
        "IN_PARTICIPOU_AVALIACAO",
        when(
            (col("IN_PRESENCA_LP") == 1) &
            (col("IN_PREENCHIMENTO_LP") == 1),
            1
        ).otherwise(0)
    )

    .withColumn(
        "DS_PARTICIPACAO",
        when(col("IN_PARTICIPOU_AVALIACAO") == 1, "Participou")
        .otherwise("Não participou")
    )

    # =====================================================
    # Situação da avaliação
    # =====================================================

    .withColumn(
        "DS_SITUACAO_AVALIACAO",
        when(
            col("VL_PROFICIENCIA_LP").isNull(),
            "Não Avaliado"
        ).otherwise("Avaliado")
    )

    # =====================================================
    # Município formatado
    # =====================================================

    .withColumn(
        "CO_MUNICIPIO_IBGE",
        lpad(col("CO_MUNICIPIO").cast("string"), 7, "0")
    )

    .withColumn(
        "NO_MUNICIPIO_UF",
        concat_ws(
            " - ",
            col("NO_MUNICIPIO"),
            col("SG_UF")
        )
    )

    # =====================================================
    # Dependência administrativa
    # =====================================================

    .withColumn(
        "DS_DEPENDENCIA",
        when(col("TP_DEPENDENCIA") == 1, "Federal")
        .when(col("TP_DEPENDENCIA") == 2, "Estadual")
        .when(col("TP_DEPENDENCIA") == 3, "Municipal")
        .when(col("TP_DEPENDENCIA") == 4, "Privada")
    )

    # =====================================================
    # Série
    # =====================================================

    .withColumn(
        "DS_SERIE",
        when(
            col("TP_SERIE") == 2,
            "2º Ano do Ensino Fundamental"
        )
    )

    # =====================================================
    # Alfabetização
    # =====================================================

    .withColumn(
        "DS_ALFABETIZADO",
        when(col("IN_ALFABETIZADO") == 1, "Sim")
        .otherwise("Não")
    )

    # =====================================================
    # Região do Brasil
    # =====================================================

    .withColumn(
        "REGIAO",
        when(
            col("SG_UF").isin(
                "AC","AP","AM","PA","RO","RR","TO"
            ),
            "Norte"
        )
        .when(
            col("SG_UF").isin(
                "AL","BA","CE","MA","PB","PE","PI","RN","SE"
            ),
            "Nordeste"
        )
        .when(
            col("SG_UF").isin(
                "DF","GO","MT","MS"
            ),
            "Centro-Oeste"
        )
        .when(
            col("SG_UF").isin(
                "ES","MG","RJ","SP"
            ),
            "Sudeste"
        )
        .when(
            col("SG_UF").isin(
                "PR","RS","SC"
            ),
            "Sul"
        )
    )

    # =====================================================
    # Auditoria
    # =====================================================

    .withColumn(
        "ANO_CARGA",
        year(col("TS_PROCESSAMENTO"))
    )

    .withColumn(
        "MES_CARGA",
        month(col("TS_PROCESSAMENTO"))
    )

)

In [0]:
df_aluno_gold_selected = df_aluno_gold.select(

    # ============================================
    # Chave Técnica
    # ============================================
    "SK_ALUNO",

    # ============================================
    # Chaves de Negócio
    # ============================================
    "NU_ANO_AVALIACAO",
    "ANO_REFERENCIA",

    "ID_ALUNO",
    "ID_ESCOLA",

    # ============================================
    # Localização
    # ============================================
    "CO_UF",
    "SG_UF",
    "REGIAO",

    "CO_MUNICIPIO",
    "CO_MUNICIPIO_IBGE",
    "NO_MUNICIPIO",
    "NO_MUNICIPIO_UF",

    # ============================================
    # Escola
    # ============================================
    "TP_DEPENDENCIA",
    "DS_DEPENDENCIA",

    "TP_SERIE",
    "DS_SERIE",

    # ============================================
    # Participação
    # ============================================
    "IN_PRESENCA_LP",
    "IN_PREENCHIMENTO_LP",

    "IN_PARTICIPOU_AVALIACAO",
    "DS_PARTICIPACAO",

    "DS_SITUACAO_AVALIACAO",

    # ============================================
    # Prova
    # ============================================
    "CO_CADERNO_LP",

    "CO_BLOCO_1",
    "TX_RESPOSTA_BLOCO_1",
    "TX_GABARITO_BLOCO_1",

    "CO_BLOCO_2",
    "TX_RESPOSTA_BLOCO_2",
    "TX_GABARITO_BLOCO_2",

    "CO_BLOCO_3",
    "TX_RESPOSTA_BLOCO_3",
    "TX_GABARITO_BLOCO_3",

    "CO_BLOCO_4",
    "TX_RESPOSTA_BLOCO_4",
    "TX_GABARITO_BLOCO_4",

    # ============================================
    # Resultado
    # ============================================
    "VL_PESO_ALUNO_LP",

    "VL_PROFICIENCIA_LP",
    "FAIXA_PROFICIENCIA",

    "IN_ALFABETIZADO",
    "DS_ALFABETIZADO",

    # ============================================
    # Auditoria
    # ============================================
    "ANO_CARGA",
    "MES_CARGA",

    "DT_PROCESSAMENTO",
    "TS_PROCESSAMENTO"
)

In [0]:
duplicados = (
    df_aluno_gold_selected
    .groupBy(
        "SK_ALUNO"
    )
    .agg(count("*").alias("QTD"))
    .filter(col("QTD") > 1)
)

if duplicados.limit(1).count() > 0:
    raise Exception("Foram encontrados registros duplicados na chave de negócio.")

In [0]:
display(
    df_aluno_gold_selected.limit(5)
)